***Mit diesem Notebook*** 
- wird die geojson Datei mit dem Baum-Kataster von Münster automatisiert auf den PC geladen
- Die geojson-Datei wird in geopandas eingelesen -> als gdf
- Dann analysiert
 - dabei fiel auf, dass nicht jede Angabe in der Baumart (Eigentlich eine Gattung) tatsächlich eine Baumart ist.
deshalb:
- Extraktion der Gattung und Erzeugung einer neuen Spalte "genus"

- Zusammenfassung mit counts für einmaligen Abgleich:
- Automatisierter Abgleich über eine API von GBIF.org (Global Biodiversity Information Facility)

- Mapping mit der geopandas gdf mit Erstellung einer neuen Spalte "is_True"

- Speicherung für DeepTress als Geopackages (baeume.gpkg)

- spätere Speicherung möglich für Faster R-CNN als COCO-JSON mit bounding boxes, hierfür kann die gpkg wieder in einen gdf (geopandasdataframe) umgewandelt werden.

            # für Faster R-CNN kann mit bounding boxes aus baeume.gpkg eine COCO-JSON gebaut werden
            # hierzu erst wieder als gdf laden
            '''
            import geopandas as gpd
            gdf = gpd.read_file("baeume.gpkg")
            '''

In [ ]:
import requests

# Baumkataster Daten für Münster als geojson laden

url = "https://geo.stadt-muenster.de/mapserv/odgruen_serv?SERVICE=WFS&VERSION=1.1.0&REQUEST=GetFeature&TYPENAME=Baeume&OUTPUTFORMAT=geojson"

r = requests.get(url)
with open("data/baumkataster_muenster.geojson", "wb") as f:
    f.write(r.content)

print("Download fertig")


In [ ]:
# in geopandas einlesen
import geopandas as gpd

gdf = gpd.read_file("data/baumkataster_muenster.geojson")
# baumprojekt_test\data\baumkataster_muenster.geojson

# Untersuchen
gdf["baumgruppe"].value_counts()

In [ ]:
gdf["baumgruppe"].value_counts().head(20)

In [ ]:
# Es gibt in der Liste der botanischen Namen offensichtlich Einträge, die KEINE lateinische Bezeichnung haben

# Um die Liste abzugleichen gibt es eine offizielle Liste, gegen die man abgleichen kann. 
# Da diese viel zu lang ist und fragmentiert, gibt es auch eine API, bei der man die einzelnen Namen eintragen kann und eine Antwort bekommt. 
# 
# Funktioniert gut, wenn mit kingdom und genus eingegrenzt wird.

***Nachfolgend kommt die eigendliche Bearbeitung der Daten:***

In [ ]:
# --------------------------------------------------------------------------
# GESAMTPROZESS: GATTUNG EXTRAHIEREN → ZUSAMMENFASSUNG → API-PRÜFUNG → MAPPING
# --------------------------------------------------------------------------
# Dieser Code:
# 1) extrahiert aus "baumgruppe" die botanische Gattung (Spalte: genus)
# 2) erzeugt eine Zusammenfassung aller eindeutigen Gattungen (DataFrame: genus_counts)
# 3) prüft jede Gattung einmal über die GBIF-API (Spalte: is_True)
# 4) mappt das Ergebnis zurück ins gdf (neue Spalte: is_True)
# --------------------------------------------------------------------------


# --------------------------------------------------------------------------
# 1) FUNKTION: Gattung extrahieren
#    - entfernt Leerzeichen 
#    - schneidet Bindestriche und Folgendes ab (Wie z.B. bei Malus-Hybride)
#    - nimmt nur das erste Wort (Gattung besteht aus 1 Wort, Art setzt sich zusammen aus Gattung und zusätzlichem Namen plus evtl. Kürzel für Entdecker)
#    - gibt "n.d." zurück, wenn keine Gattung extrahiert werden kann
# --------------------------------------------------------------------------

def extract_genus(name: str) -> str:
    if not isinstance(name, str) or not name.strip():
        return "n.d."

    clean = name.strip()

    # Bindestrich abschneiden
    if "-" in clean:
        clean = clean.split("-")[0].strip()

    # Nur erstes Wort behalten
    parts = clean.split()
    if not parts:
        return "n.d."

    return parts[0]


# --------------------------------------------------------------------------
# 2) NEUE SPALTE: genus
#    - wird direkt im gdf erzeugt
#    - Grundlage für alle weiteren Schritte
# --------------------------------------------------------------------------

gdf["genus"] = gdf["baumgruppe"].apply(extract_genus)
print("Spalte 'genus' wurde erzeugt.")


# --------------------------------------------------------------------------
# 3) ZUSAMMENFASSUNG: genus_counts
#    - zählt jede Gattung einmal
#    - erzeugt Spalte 'anzahl'
#    - erzeugt Spalte 'is_True' (wird später gefüllt)
# --------------------------------------------------------------------------

genus_counts = (
    gdf["genus"]
    .value_counts()
    .rename("anzahl")
    .to_frame()
)

genus_counts["is_True"] = False
print("DataFrame 'genus_counts' wurde erzeugt.")


# --------------------------------------------------------------------------
# 4) FUNKTION: API-Prüfung einer Gattung
#    - prüft nur die Gattung, nicht den Originalnamen
#    - nutzt GBIF-API: rank=GENUS, kingdomKey=6 (Pflanzen)
# --------------------------------------------------------------------------

import requests

def check_genus(genus: str) -> bool:
    if genus == "n.d.":
        return False

    url = (
        "https://api.gbif.org/v1/species/search"
        f"?q={genus}&rank=GENUS&kingdomKey=6"
    )

    try:
        data = requests.get(url, timeout=10).json()
    except:
        return False

    results = data.get("results", [])
    if not results:
        return False

    # exakte Übereinstimmung prüfen
    for entry in results:
        scientific = entry.get("canonicalName") or entry.get("scientificName")
        if scientific and scientific.lower() == genus.lower():
            return True

    return False


# --------------------------------------------------------------------------
# 5) API-PRÜFUNG AUF ALLE GATTUNGEN ANWENDEN
#    - jede Gattung wird genau einmal geprüft
#    - Ergebnis wird in genus_counts["is_True"] gespeichert
# --------------------------------------------------------------------------

for genus in genus_counts.index:
    result = check_genus(genus)
    genus_counts.loc[genus, "is_True"] = result
    print(f"{genus}: {result}")


# --------------------------------------------------------------------------
# 6) MANUELLE KORREKTUREN (OPTIONAL)
#    - hier kannst du einzelne Gattungen korrigieren
#    - Beispiel:
#    genus_counts.loc["Robinia", "is_True"] = True
# --------------------------------------------------------------------------

# Beispiel (auskommentiert):
# genus_counts.loc["Robinia", "is_True"] = True
# genus_counts.loc["Quercus", "is_True"] = True


# --------------------------------------------------------------------------
# 7) MAPPING ZURÜCK INS GDF
#    - gdf["is_True"] erhält das API-Ergebnis pro Gattung
# --------------------------------------------------------------------------

gdf["is_True"] = gdf["genus"].map(genus_counts["is_True"])
print("Spalte 'is_True' wurde im gdf erzeugt.")


# --------------------------------------------------------------------------
# 8) KONTROLLE (OPTIONAL)
# --------------------------------------------------------------------------

# print(gdf[["baumgruppe", "genus", "is_True"]].head())
# print(genus_counts)
# print(genus_counts[genus_counts["is_True"] == False])


***Testen nach Bearbeitung:***

In [ ]:
# nochmal Untersuchen nach Überprüfung der Arten und Mapping
gdf["baumgruppe"].value_counts()

In [ ]:
# nur die mit Baumart "is_True"

gdf[gdf["is_True"] == True]["baumgruppe"].value_counts()

In [ ]:
gdf[gdf["is_True"] == False]["baumgruppe"].value_counts()

***Sichern:***

In [ ]:
# für DeepTrees als geopackage speichern
gdf.to_file("baeume.gpkg", driver="GPKG")
print("baeume.gpkg ist erstellt")


***Wie kann es weiter gehen:***

In [ ]:
# für Faster R-CNN kann mit bounding boxes aus baeume.gpkg eine COCO-JSON gebaut werden
# hierzu erst wieder als gdf laden
'''
import geopandas as gpd
gdf = gpd.read_file("baeume.gpkg")
'''